Hugging Face Transformers & Pipelines

### AI Builders Bootcamp

Welcome! In this notebook you'll open the hood on the **Hugging Face `transformers` library** —
the single most important tool for working with modern language models.

By the end you will be able to:

- Explain what `transformers`, `Model`, `Tokenizer`, `Config`, and `Pipeline` actually are
- Run 8 different NLP tasks with **one line of code** using `pipeline(...)`
- Reproduce a pipeline's output **manually**, step by step, so nothing feels like magic
- Load models with the right `AutoModelFor...` class for the job
- Control text generation with `max_new_tokens`, `temperature`, `top_k`, `top_p`, `do_sample`, `repetition_penalty`

Let's go! 🚀


## 1. What is Hugging Face? What is `transformers`?

**Hugging Face** is a company and open community that hosts the **Hugging Face Hub**: a public
registry of

- **Models** — pretrained weights (BERT, GPT-2, Llama, Gemma, Mistral, Qwen, ...)
- **Datasets** — ready-to-use training/eval data
- **Spaces** — hosted demo apps

**`transformers`** is their Python library that gives every one of those models the **same
interface**. Before `transformers`, every research group shipped its own code for loading a model
and running it — using BERT and using GPT-2 meant learning two completely different codebases.
`transformers` standardized this: once you know the pattern, you know it for (almost) every model
on the Hub.

### Why is it "the standard library" for LLMs?

- **One API, thousands of models.** Swap `"bert-base-uncased"` for `"gpt2"` or `"meta-llama/Llama-3.2-1B"`
  and 95% of your code doesn't change.
- **Interoperable with PyTorch, TensorFlow, and JAX.**
- **Battle-tested** building blocks: tokenization, generation, training loops (`Trainer`), quantization,
  distributed inference.
- It's the **substrate** other tools are built on: `sentence-transformers`, `peft`, `trl`, `diffusers`,
  and most RAG/agent frameworks call into `transformers` under the hood.

## 1.1 Four core concepts

Every model on the Hub is really a combination of four pieces. Getting these straight now will save
you hours of confusion later.

| Concept | What it is | Analogy |
|---|---|---|
| **Config** | A JSON file describing the model's *architecture* — number of layers, hidden size, vocab size, etc. No weights, just the blueprint. | The architectural blueprint of a house |
| **Tokenizer** | Converts raw text ↔ integers (token IDs) that the model can consume. Comes with its own vocabulary. | A translator between "human language" and "model language" |
| **Model** | The neural network itself: layers of weights that transform token IDs into predictions (logits). | The house, built from the blueprint |
| **Pipeline** | A high-level wrapper that chains Tokenizer → Model → post-processing into one callable. | The furnished, move-in-ready house — you don't see the plumbing |

We'll meet all four today, working from the friendliest (Pipeline) down to the most explicit
(Config, raw tensors).


## 2. Installing dependencies

We need:

- `transformers` — the star of the show
- `datasets` — for loading example data later in the bootcamp
- `tokenizers` — the fast, Rust-backed tokenization engine `transformers` uses under the hood
- `accelerate` — lets `transformers` place models on GPU/CPU/multi-device automatically
- `sentencepiece` — required by some tokenizers (T5, Llama, Gemma, ...)
- `torch` — the deep learning framework we'll use as the backend

Colab already has `torch` preinstalled, so this cell mostly installs the Hugging Face stack.


In [ ]:
!pip install -q "transformers>=4.46" datasets tokenizers accelerate sentencepiece
import transformers
print("transformers version:", transformers.__version__)


transformers version: 5.12.1


## 3. Your first pipeline

The `pipeline()` function is the fastest way to run a model. You give it a **task name**, it picks
a sensible default model, downloads it, and hands you back a callable.

```
result = pipeline("task-name")(your_input)
```

Let's start with **sentiment analysis**.


In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result = classifier("I absolutely loved this movie, the acting was superb!")
print(result)

result2 = classifier([
    "The plot was predictable and the pacing dragged.",
    "A masterpiece from start to finish.",
])
print(result2)


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998788833618164}]
[{'label': 'NEGATIVE', 'score': 0.9998164772987366}, {'label': 'POSITIVE', 'score': 0.9998683929443359}]


### 🧠 Discuss
The output is a list of dicts with a `label` and a `score`. Where do you think that `label`
(`POSITIVE` / `NEGATIVE`) comes from — is it something the model "knows" innately, or something
baked in by whoever trained it?

> It's the second one: the label names come from the **dataset the model was fine-tuned on**. A
> differently-trained sentiment model might use `LABEL_0` / `LABEL_1`, or a 1–5 star scale.

---

Now let's run the same pattern across the other core NLP tasks. Notice how **the shape of the
code never changes** — only the task name and the input format.


### 3.1 Text classification (topic labels)

In [ ]:
topic_classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")
print(topic_classifier("The stock market rallied today after strong earnings reports."))


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9989480376243591}]


### 3.2 Text generation

In [ ]:
generator = pipeline("text-generation", model="gpt2")
output = generator(
    "In the future, artificial intelligence will",
    max_new_tokens=50,
    num_return_sequences=1,
)
print(output[0]["generated_text"])


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the future, artificial intelligence will enable us to get to know and help people. It will allow us to do things like help people in emergency situations. It will enable us to find the right people for our needs. It will enable us to help people when there is a shortage of


### 3.6 Fill-mask

In [ ]:
unmasker = pipeline("fill-mask", model="bert-base-uncased")
for pred in unmasker("The capital of France is [MASK]."):
    print(f"{pred['token_str']:>15}  score={pred['score']:.3f}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

          paris  score=0.417
          lille  score=0.071
           lyon  score=0.063
      marseille  score=0.044
          tours  score=0.030


### 3.7 Zero-shot classification

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

result = zero_shot(
    "This new smartphone has a stunning display and all-day battery life.",
    candidate_labels=["technology", "sports", "politics", "food"],
)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label:>12}: {score:.3f}")


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

  technology: 0.985
      sports: 0.012
    politics: 0.002
        food: 0.002


### 3.8 Named entity recognition (NER)

In [ ]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

text = "Sundar Pichai is the CEO of Google, headquartered in Mountain View, California."
for entity in ner(text):
    print(f"{entity['word']:>15}  {entity['entity_group']:>5}  score={entity['score']:.3f}")


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

  Sundar Pichai    PER  score=0.990
         Google    ORG  score=0.999
  Mountain View    LOC  score=0.998
     California    LOC  score=0.999


### 👉 TODO — try your own input
Pick **two** of the pipelines above and re-run them with a sentence of your own. Does the model
behave the way you expect? Try to find an input that "breaks" it (ambiguous sentiment, a trick
question for QA, etc.).


In [ ]:
# Your experiment here


## 4. Behind the pipeline

`pipeline("sentiment-analysis")(text)` feels like one step, but it's actually **four**:

```
Raw text
   │
   ▼
Tokenizer  ──►  Input IDs + Attention Mask
   │
   ▼
Model  ──►  Logits (raw, unnormalized scores)
   │
   ▼
Post-processing  ──►  softmax → label + confidence score
```

Let's reproduce sentiment analysis **by hand**, one arrow at a time, using the exact model the
pipeline used by default: `distilbert-base-uncased-finetuned-sst-2-english`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "I absolutely loved this movie, the acting was superb!"


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

**Step 1 — Tokenizer: text → input IDs + attention mask**

In [ ]:
encoded = tokenizer(text, return_tensors="pt")
print("input_ids:     ", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print()
print("Decoded tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))


input_ids:      tensor([[  101,  1045,  7078,  3866,  2023,  3185,  1010,  1996,  3772,  2001,
         21688,   999,   102]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Decoded tokens: ['[CLS]', 'i', 'absolutely', 'loved', 'this', 'movie', ',', 'the', 'acting', 'was', 'superb', '!', '[SEP]']


**Step 2 — Model: input IDs → logits**

The model doesn't output "POSITIVE" or "0.99" — it outputs raw, unbounded numbers called **logits**,
one per class.


In [ ]:
with torch.no_grad():
    outputs = model(**encoded)

logits = outputs.logits
print("Logits:", logits)
print("Model's label map:", model.config.id2label)


Logits: tensor([[-4.3272,  4.6913]])
Model's label map: {0: 'NEGATIVE', 1: 'POSITIVE'}


**Step 3 — Post-processing: logits → human-readable output**

We turn logits into probabilities with **softmax**, then pick the highest one.


In [ ]:
import torch.nn.functional as F

probs = F.softmax(logits, dim=-1)
predicted_class = probs.argmax(dim=-1).item()

print("Probabilities:", probs)
print("Predicted label:", model.config.id2label[predicted_class])
print("Confidence:", probs[0, predicted_class].item())


Probabilities: tensor([[1.2113e-04, 9.9988e-01]])
Predicted label: POSITIVE
Confidence: 0.9998788833618164


### ✅ Checkpoint
This manual result should match `pipeline("sentiment-analysis")(text)` from Section 3 exactly —
same label, same score (up to floating-point rounding). That's the whole trick behind
`pipeline`: **tokenize → model → softmax/argmax → look up label**.

### 🧠 Discuss
Why do you think `transformers` returns raw **logits** from the model instead of directly returning
probabilities?

> Logits are more numerically flexible: you can apply *different* post-processing on top of them
> depending on the task (softmax for classification, sigmoid for multi-label, temperature-scaled
> softmax for generation, greedy argmax for token prediction...). Baking softmax into the model
> would lock in one interpretation.


## 5. Loading models manually — the `Auto*` classes

You just used `AutoModelForSequenceClassification`. `transformers` has a whole family of
`AutoModelFor...` classes — each one adds a different "head" on top of the same base transformer,
tailored to a different task shape.

| Class | Adds on top of the base model | Used for |
|---|---|---|
| `AutoModel` | Nothing — returns raw hidden states | Embeddings, feature extraction, building custom heads |
| `AutoModelForCausalLM` | A next-token prediction head | Text generation (GPT-2, Llama, Mistral, Gemma...) |
| `AutoModelForSequenceClassification` | A classification head over the whole sequence | Sentiment, topic classification, NLI |
| `AutoModelForSeq2SeqLM` | An encoder-decoder generation head | Translation, summarization (T5, BART, Marian) |

`AutoTokenizer` works the same way for all of them — it inspects the model's config and hub files
and returns the right tokenizer class automatically (`BertTokenizerFast`, `GPT2TokenizerFast`, ...).

Let's load one of each and compare.


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
)

# AutoModel: just the base transformer, no task head
base_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
base_model = AutoModel.from_pretrained("bert-base-uncased")
print("AutoModel output type:", type(base_model).__name__)

# AutoModelForCausalLM: for text generation
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")
print("AutoModelForCausalLM output type:", type(gpt2_model).__name__)

# AutoModelForSeq2SeqLM: for translation/summarization
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
print("AutoModelForSeq2SeqLM output type:", type(t5_model).__name__)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AutoModel output type: BertModel


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

AutoModelForCausalLM output type: GPT2LMHeadModel


...

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

AutoModelForSeq2SeqLM output type: T5ForConditionalGeneration


### 👉 TODO
Look at the printed class names. `AutoModelForCausalLM` resolved to `GPT2LMHeadModel` — a
GPT-2-specific class. Why do you think `transformers` needs a different concrete class per
architecture, even though you request them all through the same `AutoModelForCausalLM` name?

💡 **Hint:** think about what differs between GPT-2's internals and, say, Llama's internals
(positional encodings, attention implementation, layer norm placement...) versus what stays the
same at the *interface* level (`.generate()`, `.forward()`, `input_ids` in, logits out).


## 6. Tokenizer + Model, together, manually

Let's slow down even further and inspect every tensor that flows between tokenizer and model,
this time with `AutoModel` (no task head) so we can see raw **hidden states** — the contextual
representation of every token.


In [ ]:
text = "Hugging Face makes NLP accessible."

inputs = base_tokenizer(text, return_tensors="pt")
print("Keys returned by the tokenizer:", list(inputs.keys()))
print("input_ids shape:", inputs["input_ids"].shape)
print("Tokens:", base_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))


Keys returned by the tokenizer: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids shape: torch.Size([1, 9])
Tokens: ['[CLS]', 'hugging', 'face', 'makes', 'nl', '##p', 'accessible', '.', '[SEP]']


In [ ]:
with torch.no_grad():
    outputs = base_model(**inputs)

last_hidden_state = outputs.last_hidden_state
print("last_hidden_state shape:", last_hidden_state.shape)
print("  -> (batch_size, sequence_length, hidden_size)")
print()
print("Vector for the [CLS] token (first 10 dims):", last_hidden_state[0, 0, :10])


last_hidden_state shape: torch.Size([1, 9, 768])
  -> (batch_size, sequence_length, hidden_size)

Vector for the [CLS] token (first 10 dims): tensor([-0.5522, -0.0982,  0.0775, -0.1996, -0.4380, -0.1550,  0.1928,  0.1775,
        -0.1394, -0.4005])


### 🧠 Discuss
Notice the shape: `(1, 8, 768)`. Every **token**, not just the sentence as a whole, gets its own
768-dimensional vector. This is why the same word can have a *different* embedding depending on
its context — the vector for "bank" in "river bank" differs from "bank" in "money bank", because
each token's vector is computed *with attention to its neighbors*.


### 👉 TODO — decode individual tokens
Loop over `inputs["input_ids"][0]` and print each token decoded individually, matching what we did
with `convert_ids_to_tokens` but using `.decode()` instead. Do you get the same strings?


In [ ]:
for token_id in inputs["input_ids"][0]:
    print(base_tokenizer.decode(token_id))


## 7. Text generation with `.generate()`

`AutoModelForCausalLM` models expose a `.generate()` method that repeatedly predicts the next
token and appends it, until it hits a stop condition. Every generation parameter you'll ever tune
controls **one part of this loop**.

```
prompt tokens
     │
     ▼
 model(...)  ──►  logits for next token
     │
     ▼
 sampling strategy (greedy / top-k / top-p / temperature)
     │
     ▼
 pick next token id  ──►  append to sequence  ──►  repeat until max_new_tokens or eos_token_id
```


In [ ]:
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # GPT-2 has no PAD token by default

prompt = "The best way to learn machine learning is"
inputs = gpt2_tokenizer(prompt, return_tensors="pt")

output_ids = gpt2_model.generate(
    **inputs,
    max_new_tokens=25,
    do_sample=False,          # greedy decoding: always pick the single most likely token
    pad_token_id=gpt2_tokenizer.eos_token_id,
)
print(gpt2_tokenizer.decode(output_ids[0], skip_special_tokens=True))


The best way to learn machine learning is to learn it from the ground up.

The best way to learn machine learning is to learn it from the ground up


### 7.1 Generation parameters, one at a time

Let's change **one parameter at a time** and observe the effect. Same prompt, same model, every
time — only the decoding strategy changes.


In [ ]:
def generate_with(**gen_kwargs):
    output_ids = gpt2_model.generate(
        **inputs,
        pad_token_id=gpt2_tokenizer.eos_token_id,
        **gen_kwargs,
    )
    return gpt2_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("do_sample=False (greedy):")
print(" ", generate_with(max_new_tokens=25, do_sample=False))
print()

torch.manual_seed(0)
print("do_sample=True, temperature=0.7:")
print(" ", generate_with(max_new_tokens=25, do_sample=True, temperature=0.7))
print()

torch.manual_seed(0)
print("do_sample=True, temperature=1.5 (more random):")
print(" ", generate_with(max_new_tokens=25, do_sample=True, temperature=1.5))


do_sample=False (greedy):
  The best way to learn machine learning is to learn it from the ground up.

The best way to learn machine learning is to learn it from the ground up

do_sample=True, temperature=0.7:
  The best way to learn machine learning is to study it. You can use the "learn" tool to see how the machine learning process works and what you can do

do_sample=True, temperature=1.5 (more random):
  The best way to learn machine learning is to study it with our customers...


| Parameter | What it controls |
|---|---|
| `max_new_tokens` | Hard cap on how many tokens to generate (independent of prompt length) |
| `do_sample` | `False` = greedy (always the top token, fully deterministic). `True` = sample from the distribution |
| `temperature` | Rescales logits before softmax. `<1.0` sharpens the distribution (more confident/repetitive), `>1.0` flattens it (more random/diverse) |
| `top_k` | Only sample from the `k` most likely next tokens |
| `top_p` (nucleus) | Only sample from the smallest set of tokens whose cumulative probability ≥ `p` |
| `repetition_penalty` | `>1.0` discourages repeating tokens already generated |
| `eos_token_id` | Token ID that means "stop generating" |
| `pad_token_id` | Token ID used to pad shorter sequences in a batch |


In [ ]:
torch.manual_seed(0)
print("top_k=5:")
print(" ", generate_with(max_new_tokens=25, do_sample=True, top_k=5))
print()

torch.manual_seed(0)
print("top_p=0.9:")
print(" ", generate_with(max_new_tokens=25, do_sample=True, top_p=0.9, top_k=0))
print()

print("repetition_penalty=1.0 (off) vs 1.8 (strong) with greedy + a repetition-prone prompt:")
repeat_inputs = gpt2_tokenizer("I like to repeat myself. I like to repeat myself.", return_tensors="pt")
out_no_penalty = gpt2_model.generate(**repeat_inputs, max_new_tokens=20, do_sample=False,
                                     pad_token_id=gpt2_tokenizer.eos_token_id, repetition_penalty=1.0)
out_with_penalty = gpt2_model.generate(**repeat_inputs, max_new_tokens=20, do_sample=False,
                                       pad_token_id=gpt2_tokenizer.eos_token_id, repetition_penalty=1.8)
print(" no penalty:  ", gpt2_tokenizer.decode(out_no_penalty[0], skip_special_tokens=True))
print(" penalty=1.8: ", gpt2_tokenizer.decode(out_with_penalty[0], skip_special_tokens=True))


top_k=5:
  The best way to learn machine learning is to go to the top of your school, and to get the basics right.

This is the best way to do

top_p=0.9:
  The best way to learn machine learning is to study it. You need to test it out for yourself and then show the machine learning principles that people like to test out

repetition_penalty=1.0 (off) vs 1.8 (strong) with greedy + a repetition-prone prompt:
 no penalty:   I like to repeat myself. I like to repeat myself. I like to repeat myself. I like to repeat myself. I like to repeat myself. I like
 penalty=1.8:  I like to repeat myself. I like to repeat myself.
The first time, it was a little bit of an embarrassment for me because my dad had been


### 👉 TODO — find your own "sweet spot"
Try `temperature` values of `0.2`, `1.0`, and `2.0` with `do_sample=True` on a prompt of your
choice. At what point does the output stop making sense?

### 🧠 Discuss
Why is `do_sample=False` (greedy) **fully deterministic** across runs, while `do_sample=True`
requires `torch.manual_seed(...)` to be reproducible?


In [ ]:
# Your experiment here


## 8. Different tasks, different models — why so many?

Let's load **three encoder models** and see how their pretraining objective shows up in what
they're good at.


In [ ]:
text = "The movie was a [MASK] experience from start to finish."

bert_fill = pipeline("fill-mask", model="bert-base-uncased")
roberta_fill = pipeline("fill-mask", model="roberta-base")

print("BERT top guesses:")
for p in bert_fill(text.replace("[MASK]", bert_fill.tokenizer.mask_token)):
    print(" ", p["token_str"], round(p["score"], 3))

# RoBERTa uses a different mask token (<mask>) and different tokenization (byte-level BPE)
roberta_text = "The movie was a <mask> experience from start to finish."
print("\nRoBERTa top guesses:")
for p in roberta_fill(roberta_text):
    print(" ", p["token_str"], round(p["score"], 3))


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

BERT top guesses:
  great 0.181
  fantastic 0.094
  wonderful 0.062
  tremendous 0.039
  fun 0.035

RoBERTa top guesses:
   great 0.222
   learning 0.096
   thrilling 0.052
   fantastic 0.044
   wonderful 0.043


### 🧠 Discuss
BERT and RoBERTa are both **encoder-only masked language models**, trained the same way in
principle — yet they use different mask tokens (`[MASK]` vs `<mask>`) and different tokenizers
(WordPiece vs byte-level BPE). This is your first hint of what Notebook 2 is entirely about:
**the tokenizer is not a detail — it's load-bearing, model-specific infrastructure.**

### Recap: which `AutoModelFor...` / task fits which job?

| Job | Class | Example models |
|---|---|---|
| "Is this positive or negative?" | `AutoModelForSequenceClassification` | DistilBERT-SST2, RoBERTa-MNLI |
| "Fill in this blank" | `AutoModelForMaskedLM` (used internally by `fill-mask`) | BERT, RoBERTa |
| "Continue this text" | `AutoModelForCausalLM` | GPT-2, Llama, Mistral, Gemma, Qwen |
| "Translate / summarize this" | `AutoModelForSeq2SeqLM` | T5, BART, Marian |
| "Just give me embeddings" | `AutoModel` | Any encoder, for custom downstream use |


---
## ✅ Wrap-up — what you learned

- `transformers` gives every model on the Hub the **same interface**: Config (blueprint) +
  Tokenizer (translator) + Model (the network) + Pipeline (the packaged product).
- `pipeline(task)` is really: **tokenize → model forward pass → post-process logits → readable
  output** — and you reproduced every one of those steps by hand.
- The right `AutoModelFor...` class depends on the **shape of the task**: classification head,
  causal LM head, seq2seq head, or no head at all.
- Generation is a loop of "predict next token, append, repeat" — and every generation parameter
  (`temperature`, `top_k`, `top_p`, `do_sample`, `repetition_penalty`) tunes **how that next token
  gets chosen**.

### 🌟 Stretch goals
1. Swap `gpt2` for a small instruction-tuned model (e.g. `HuggingFaceTB/SmolLM2-360M-Instruct`)
   and repeat the generation experiments. How do good defaults differ?
2. Build a tiny "compare 3 sentiment models" cell: run the same 5 sentences through 3 different
   sentiment pipelines and tabulate agreement/disagreement.
3. Try `AutoModelForQuestionAnswering` directly (no pipeline) and reproduce Section 3.5 manually,
   the same way we did for sentiment analysis in Section 4.

**Next up:** Notebook 2 — Tokenizers Deep Dive, where we build a BPE tokenizer from scratch and
compare tokenizers across GPT, Llama, Gemma, BERT, RoBERTa, T5, Mistral, and Qwen.
